In [1]:
import pandas as pd
import numpy as np
import random
import torch
import os
import csv
import time
import gc
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
from itertools import combinations
import random
from gensim.models import LdaModel
import spacy

import sys
sys.path.append('./tools')
from Matave import Matave

In [2]:
K_RANGE = list(range(3, 20)) # chosen to prevent very large numbers of topics in synthetic data generation step
TOP_N = 10

nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner", "parser"]  # speed
)

In [3]:
# Random States
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
domains = {
    'yahoo': 'non-factoid question',
    'banking77': 'banking text',
    'huffPostNews': 'news',
    'clinc150': 'multi-domain intent',
    'atis': 'air travel information system',
    'medicalAbstracts': 'medical abstract (current patient condition)',
    'dementiaAudio': 'dementia cookie theft picture description',
    'syntheticCareHomeNurseNotes': 'nursing home resident',
    'clinicalDialogueSummarizations': 'clinical note',
    'simSUM': 'compact clinical note'
}

In [6]:
# remove punctuation, make lowercase, remove stopwords, punctuation, lemmatize, remove documents with less than 5 tokens
def preprocess_texts(texts, min_words = 5):
    cleaned_texts = []
    for doc in nlp.pipe(texts, batch_size=1000):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and token.lemma_ != "-PRON-"
            and token.is_alpha
        ]
        if len(tokens) >= min_words:
            cleaned_texts.append(" ".join(tokens))
    return cleaned_texts

In [7]:
def make_lda(corpus, dictionary, tokenized_texts, k):
    start = time.time()

    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=RANDOM_STATE,
        passes=10
    )

    lda_topics = [
        [word for word, _ in lda_model.show_topic(i, topn=TOP_N)]
        for i in range(k)
    ]
    end = time.time()
    coherence = get_coherence_score(lda_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(lda_topics)
    # Inverse redundancy.
    redundancy = compute_topic_redundancy(lda_topics)
    model_time = end - start
    return coherence, diversity, redundancy, model_time, lda_topics, lda_model


In [8]:
parent_path = '../getText/datasetsPrep'
all_results = []
for folder in os.listdir(f'{parent_path}'):
    if os.path.isdir(f'{parent_path}/{folder}'):
        for file in os.listdir(f'{parent_path}/{folder}'):
            if file.endswith('.csv'):
                temp_dataset_name = file.replace('.csv', '')
                df = pd.read_csv(f'{parent_path}/{folder}/{file}')
                df = df.dropna().sample(frac=1, random_state=RANDOM_STATE)
                print(f"{file}: {len(df)}")
                texts = df['text'].tolist()
                texts = preprocess_texts(texts)
                texts = random.sample(texts, 500)
                # Run MATAVE algorithm. 
                tokenized_texts = [word_tokenize(text.lower()) for text in texts]
                dictionary = Dictionary(tokenized_texts)
                corpus = [dictionary.doc2bow(text) for text in tokenized_texts]
                start = time.time()
                matave = Matave(texts, random_state = RANDOM_STATE)
                matave.fit(k_range = K_RANGE)
                matave_topics = [topic.split() for topic in matave.top_topic_words.values()]
                matave_topics = [topic[:TOP_N] for topic in matave_topics]
                end = time.time()
                matave_coherence = get_coherence_score(matave_topics, tokenized_texts, dictionary, 'c_v')
                matave_diversity = get_diversity_score(matave_topics)
                matave_redundancy = compute_topic_redundancy(matave_topics)
                matave_model_time = end - start
                topics_for_prompts_dict = {}
                for text, topic in zip(texts, matave.assigned_topics):
                    if topic in topics_for_prompts_dict:
                        topics_for_prompts_dict[topic].append(text)
                    else:
                        topics_for_prompts_dict[topic] = [text]

                matave_example_notes = []
                matave_topic_keywords = []
                for key, items in topics_for_prompts_dict.items():
                    matave_example_notes.append(random.choice(items))
                    matave_topic_keywords.append(f"Other topic keywords: {matave.top_topic_words[key]}")

                for_prompts_dict = {'topic_model': 'MATAVE', 'coherence': matave_coherence, 'diversity': matave_diversity, 'redundancy': matave_redundancy, 'time': matave_model_time,'dataset': temp_dataset_name, 'domain': domains[temp_dataset_name], 'example_notes': '*** SEPARATION ***'.join(matave_example_notes), 'topic_keywords': '\n'.join(matave_topic_keywords)}
                all_results_df = pd.DataFrame([for_prompts_dict])
                all_results_df.to_csv('./promptDataPreparation.csv', mode='a', header=not os.path.exists('./promptDataPreparation.csv'), index=False, quoting=csv.QUOTE_ALL)
                print(f"{file}: MATAVE done.")
                # Run LDA algorithm.
                lda_temp = {'k': [], 'coherence': [], 'diversity': [], 'redundancy': [], 'combined': [], 'time': []}
                for k in K_RANGE:

                        lda_coherence, lda_diversity, lda_redundancy, lda_model_time, _, _ = make_lda(corpus, dictionary, tokenized_texts, k)
                        lda_temp['k'].append(k)
                        lda_temp['coherence'].append(lda_coherence)
                        lda_temp['diversity'].append(lda_diversity)
                        lda_temp['redundancy'].append(lda_redundancy)
                        lda_temp['combined'].append((lda_coherence + lda_diversity + lda_redundancy) / 3)
                        lda_temp['time'].append(lda_model_time)

                chosen_k = lda_temp['k'][lda_temp['combined'].index(max(lda_temp['combined']))]
                lda_coherence, lda_diversity, lda_redundancy, lda_model_time, lda_topics, lda_model = make_lda(corpus, dictionary, tokenized_texts, chosen_k)
                doc_topics = [lda_model.get_document_topics(doc) for doc in corpus]
                assigned_topics = [
                    max(topics, key=lambda x: x[1])[0] if topics else None
                    for topics in doc_topics
                ]

                topics_for_prompts_dict = {}
                for text, topic in zip(texts, assigned_topics):
                    if topic in topics_for_prompts_dict:
                        topics_for_prompts_dict[topic].append(text)
                    else:
                        topics_for_prompts_dict[topic] = [text]

                lda_example_notes = []
                lda_topic_keywords = []
                for key, items in topics_for_prompts_dict.items():
                    lda_example_notes.append(random.choice(items))
                    lda_topic_keywords.append(f"Other topic keywords: {' '.join(lda_topics[key])}")

                for_prompts_dict = {'topic_model': 'LDA', 'coherence': lda_coherence, 'diversity': lda_diversity, 'redundancy': lda_redundancy, 'time': lda_model_time,'dataset': temp_dataset_name, 'domain': domains[temp_dataset_name], 'example_notes': '*** SEPARATION ***'.join(lda_example_notes), 'topic_keywords': '\n'.join(lda_topic_keywords)}
                all_results_df = pd.DataFrame([for_prompts_dict])
                all_results_df.to_csv('./promptDataPreparation.csv', mode='a', header=not os.path.exists('./promptDataPreparation.csv'), index=False, quoting=csv.QUOTE_ALL)
                print(f"{file}: LDA done.")
                del df, texts, tokenized_texts, dictionary, corpus, matave, lda_model
                gc.collect() 


yahoo.csv: 87362


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

yahoo.csv: MATAVE done.
yahoo.csv: LDA done.
banking77.csv: 13069


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

/Users/isabel/anaconda3/envs/localSyntheticData/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


simSUM.csv: MATAVE done.
simSUM.csv: LDA done.
